In [ ]:
# ========== 1. SQLite 票务库：建表 + 写入种子航班数据 ==========

# 导入 sqlite3：用本地文件数据库存票价，无需额外服务
import sqlite3

# 连接（或创建）tickets.db；check_same_thread=False 方便 Gradio 多线程读写
conn = sqlite3.connect("tickets.db", check_same_thread=False)
# 取得游标：后续 execute / executemany 都通过它
cursor = conn.cursor()

# 若表不存在则创建 tickets：自增 id + 航司名 + 出发/到达 + 价格
cursor.execute("""
CREATE TABLE IF NOT EXISTS tickets (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    source TEXT,
    destination TEXT,
    price REAL
)
""")

# 每次运行先清空表，保证演示数据可重复、可预期
cursor.execute("DELETE FROM tickets")

# 三元组种子数据：(航司, 出发地, 目的地, 价格)
data = [
    ("FlightAI", "Mumbai", "Delhi", 5000),
    ("FlightAI", "Mumbai", "Tokyo", 45000),
    ("FlightAI", "Delhi", "Bangalore", 4000),
]

# 批量插入；? 占位符防 SQL 注入
cursor.executemany(
    "INSERT INTO tickets (name, source, destination, price) VALUES (?, ?, ?, ?)",
    data
)

# 提交事务，让写入落盘
conn.commit()


In [ ]:
# ========== 2. 业务函数：查价 / 订票 / 列出全部航线 ==========

# normalize：去首尾空白并把单词 Title Case，便于和库里城市名对齐
def normalize(text): return str(text).strip().title()

# get_price：按出发地+目的地查询航司名与价格
def get_price(source, destination):
    # 参数化查询；城市名先 normalize
    cursor.execute("SELECT name, price FROM tickets WHERE source=? AND destination=?", (normalize(source), normalize(destination)))
    # fetchall：可能多行；此处种子数据通常 0 或 1 行
    return cursor.fetchall()

# book_ticket：查到航线则返回 SUCCESS 文案，否则 Error 文案（字符串保持原样）
def book_ticket(name, source, destination):
    # 先查价
    res = get_price(source, destination)
    # 无航班：错误信息保持英文原样（Agent 会检查 "SUCCESS" 子串）
    if not res: return f"Error: No flights from {source} to {destination}."
    # 取第一行的航司与价格，拼成功消息
    return f"SUCCESS: Booked {name} on {res[0][0]} for ₹{res[0][1]}"

# get_all_flights：把库里所有航线格式化成多行文本（当前 Agent 主路径未必调用）
def get_all_flights():
    """返回所有可用航线与价格的格式化字符串。"""
    # 查出 source/destination/price
    cursor.execute("SELECT source, destination, price FROM tickets")
    rows = cursor.fetchall()
    # 空库提示（文案保持原样）
    if not rows:
        return "No flights available in the database."
    
    # 每行一条「✈️ A to B: ₹price」
    lines = [f"✈️ {src} to {dst}: ₹{price}" for src, dst, price in rows]
    # 用换行拼成一整段
    return "\n".join(lines)


In [ ]:
# ========== 3. OpenAI 兼容客户端指向本地 Ollama + JSON-only system prompt ==========

# 从 openai 导入 OpenAI：用官方 SDK 调「OpenAI 兼容」接口
from openai import OpenAI
# 导入 json：后面要把模型输出 parse 成字典
import json
# 变量名仍叫 ollama：base_url 指向本机 11434 的 /v1；api_key 占位即可
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# system_prompt：强制模型只输出 JSON（工具调用或最终回复）；英文原文必须保留
system_prompt = """
You are a Ticket Booking Bot. 
You MUST speak ONLY in JSON. NO COMMENTS. NO CHAT.

FORMAT 1 (Tool): {"tool": "book_ticket", "args": {"name": "Jon", "source": "Mumbai", "destination": "Tokyo"}}
FORMAT 2 (Final): {"final": "I have booked your ticket!"}

ALLOWED CITIES: Mumbai, Delhi, Tokyo, Bangalore.
If you are missing the user's Name or Source, ASK FOR IT using the 'final' format.
"""





In [ ]:
# ========== 4. agent_with_logs：最多 4 步的「模型 ↔ 工具」循环，并记录思考日志 ==========

# 定义 Agent：输入用户自然语言，返回 (给用户的答案, 日志文本)
def agent_with_logs(user_input):
    # 初始 messages：system 定规矩，user 放本轮请求
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_input}]
    # 逐步追加的可读日志（给 Gradio 侧栏展示）
    thought_log = []
    
    # 最多迭代 4 轮，防止工具调用死循环
    for i in range(4): 
        try:
            # 调本地 llama3.2:1b（模型 id 保持原样）
            response = ollama.chat.completions.create(model="llama3.2:1b", messages=messages)
            # 取出助手文本并去首尾空白
            reply = response.choices[0].message.content.strip()
            
            # 把原始回复写入日志（展示文案保持原样）
            thought_log.append(f"🤖 Step {i+1} RAW RESPONSE:\n{reply}\n{'-'*20}")
            
            # 去掉 // 行注释后，用正则抠出第一个 {...} JSON 对象
            reply_clean = re.sub(r"//.*", "", reply)
            match = re.search(r"\{.*\}", reply_clean, re.DOTALL)
            
            # 没找到 JSON：追加纠正提示并 continue 重试
            if not match: 
                thought_log.append(f"⚠️ Step {i+1}: No JSON found in raw text. Retrying...")
                messages.append({"role": "user", "content": "Please respond ONLY with a JSON object."})
                continue
            
            # 解析 JSON 字符串为 dict
            data = json.loads(match.group(0))

            # 分支 A：模型请求调用工具 book_ticket
            if "tool" in data:
                # 工具名转小写（容错）
                t_name = str(data.get("tool", "")).lower()
                # 取出参数字典
                args = data.get("args", {})
                
                # 姓名：兼容 name / traveler_name，缺省 Guest
                u_name = args.get("name") or args.get("traveler_name") or "Guest"
                # 出发地：兼容 source / src
                u_src = args.get("source") or args.get("src")
                # 目的地：兼容 destination / dst
                u_dst = args.get("destination") or args.get("dst")
                
                # 真正执行订票函数，得到 Observation 字符串
                obs = book_ticket(u_name, u_src, u_dst)
                # 把动作与结果写入日志
                thought_log.append(f"🛠️ Step {i+1} ACTION: Calling {t_name} -> Result: {obs}")
                
                # 订票成功：直接结束循环，把 SUCCESS 文案返回给用户
                if "SUCCESS" in obs:
                    thought_log.append("✅ Booking confirmed. Closing loop.")
                    return obs, "\n".join(thought_log)

                # 失败：把模型原回复 + Observation 回灌 messages，让模型下一轮改主意
                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user", "content": f"Observation: {obs}"})
                continue

            # 分支 B：模型给出最终自然语言（包在 final 字段）
            if "final" in data: 
                thought_log.append(f"🏁 Step {i+1}: Final Answer reached.")
                return data["final"], "\n".join(thought_log)

        except Exception as e:
            # JSON 解析失败等：记日志并让模型修正语法
            thought_log.append(f"❌ Step {i+1} PARSE ERROR: {str(e)}")
            messages.append({"role": "user", "content": f"JSON error: {str(e)}. Fix your syntax."})
    
    # 4 步仍未成功：返回兜底文案（保持原样）
    return "I couldn't complete the booking.", "\n".join(thought_log)


In [ ]:
# ========== 5. Gradio UI：聊天框 + Agent 思考日志并排 ==========

# chat_fn：Gradio 回调；调用 Agent，更新 history 与日志框
def chat_fn(message, history):
    # 跑 Agent，拿到给用户的答案与逐步日志
    answer, logs = agent_with_logs(message)
    # 把本轮问答追加到 Chatbot 历史（元组格式）
    history.append((message, answer))
    # 返回：清空输入框、新 history、日志文本
    return "", history, logs

# 用 Blocks 搭两栏布局；主题 Soft
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # 页面标题 Markdown（文案保持原样）
    gr.Markdown("# ✈️ AI Ticket Agent with Live Logs")
    
    # 一行两列：左聊天，右日志
    with gr.Row():
        with gr.Column(scale=2):
            # 聊天窗口高度 400
            chatbot = gr.Chatbot(height=400)
            # 用户输入框；placeholder/label 保持英文原样
            msg = gr.Textbox(placeholder="Book Jon from Mumbai to Tokyo", label="Your Request")
        with gr.Column(scale=1):
            # 只读日志框：展示 thought_log
            log_box = gr.Textbox(label="Agent Thought Process", interactive=False, lines=15)

    # 回车提交：输入 → chat_fn → 更新三处输出
    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot, log_box])

# 启动 Gradio 应用（默认本地端口）
demo.launch()
